# Cuaderno de Datos — Skills integrados a Gemini

**Actividad de Seguimiento · Ámbito: Ingeniería de Sistemas / TI**

Este cuaderno reúne y analiza los datos de la investigación sobre Skills para Gemini:

1. Datos del ecosistema de Skills (skills.sh).
2. Inventario real del proyecto generado por la Skill `auditor-tecnico-proyectos`.
3. Visualizaciones y conclusiones.

In [1]:
import json
from pathlib import Path

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.use("Agg")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| matplotlib", matplotlib.__version__)

pandas 3.0.6 | matplotlib 3.11.2


## 1. Ecosistema de Skills (skills.sh)

Selección de Skills instaladas y relevantes para el ejercicio profesional de TI.

In [2]:
skills = pd.DataFrame(
    [
        {"skill": "skill-creator", "repo": "anthropics/skills", "uso": "Diseñar Skills"},
        {"skill": "writing-for-agents", "repo": "mattpocock/skills", "uso": "Redactar SKILL.md/AGENTS.md"},
        {"skill": "research", "repo": "mattpocock/skills", "uso": "Investigar fuentes primarias"},
        {"skill": "code-review", "repo": "mattpocock/skills", "uso": "Revisión de código"},
        {"skill": "frontend-design", "repo": "anthropics/skills", "uso": "Diseño de landing page"},
    ]
)
skills

,skill,repo,uso
0,skill-creator,anthropics/skills,Diseñar Skills
1,writing-for-agents,mattpocock/skills,Redactar SKILL.md/AGENTS.md
2,research,mattpocock/skills,Investigar fuentes primarias
3,code-review,mattpocock/skills,Revisión de código
4,frontend-design,anthropics/skills,Diseño de landing page


### Dónde viven las Skills en Gemini

| Alcance | Ruta |
|---|---|
| Workspace | `.agents/skills/<skill>/` |
| Global | `~/.gemini/config/skills/<skill>/` |

Ciclo de vida: **Descubrimiento → Activación → Ejecución** (divulgación progresiva).

## 2. Inventario real del proyecto

Datos producidos por el escáner de la Skill.

In [3]:
ruta = Path("../examples/reporte_scan_ejemplo.json")
if not ruta.exists():
    ruta = Path("examples/reporte_scan_ejemplo.json")
scan = json.loads(ruta.read_text(encoding="utf-8"))

print("Raíz:", scan["raiz"])
print("Archivos:", scan["total_archivos"])
print("Líneas de código:", scan["total_lineas_codigo"])
print("Manifiestos:", scan["manifiestos"] or "(ninguno)")

Raíz: C:\Users\Juan\Desktop\Proyectos\IA_Seguimiento_2\skill-asistente-profesional\.agents\skills\auditor-tecnico-proyectos
Archivos: 4
Líneas de código: 303
Manifiestos: (ninguno)


In [4]:
lenguajes = (
    pd.DataFrame(
        {
            "lenguaje": list(scan["lineas_por_lenguaje"].keys()),
            "lineas": list(scan["lineas_por_lenguaje"].values()),
            "archivos": [scan["archivos_por_lenguaje"].get(k, 0) for k in scan["lineas_por_lenguaje"]],
        }
    )
    .sort_values("lineas", ascending=False)
    .reset_index(drop=True)
)
lenguajes

,lenguaje,lineas,archivos
0,Markdown,166,3
1,Python,137,1


In [5]:
ax = lenguajes.plot.bar(x="lenguaje", y="lineas", color="#12B5A5", legend=False)
ax.set_title("Líneas de código por lenguaje")
ax.set_xlabel("Lenguaje")
ax.set_ylabel("Líneas")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("lenguajes.png", dpi=120)
plt.show()

C:\Users\Juan\AppData\Local\Temp\ipykernel_25068\2504480078.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Señales de riesgo

El escáner marca hechos objetivos: `.env` versionado, posibles secretos, pendientes (`TODO/FIXME`) y archivos pesados.

In [6]:
sen = scan["senales_riesgo"]
riesgos = pd.DataFrame(
    [
        {"senal": "Archivos .env versionados", "valor": len(sen["env_versionado"])},
        {"senal": "Posibles secretos", "valor": len(sen["posibles_secretos"])},
        {"senal": "Pendientes (TODO/FIXME)", "valor": sen["pendientes"]},
        {"senal": "Archivos pesados", "valor": len(sen["archivos_pesados"])},
    ]
)
riesgos

,senal,valor
0,Archivos .env versionados,0
1,Posibles secretos,0
2,Pendientes (TODO/FIXME),6
3,Archivos pesados,0


In [7]:
color = {0: "#12B5A5"}
colores = ["#F2A63B" if v > 0 else "#12B5A5" for v in riesgos["valor"]]
ax = riesgos.plot.bar(x="senal", y="valor", color=colores, legend=False)
ax.set_title("Señales de riesgo detectadas")
ax.set_xlabel("")
ax.set_ylabel("Conteo")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("riesgos.png", dpi=120)
plt.show()

C:\Users\Juan\AppData\Local\Temp\ipykernel_25068\1508614934.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Conclusiones del análisis

1. La Skill genera un inventario **determinista y reproducible** sobre el que fundar el criterio.
2. El ecosistema de Skills (skills.sh) permite **instalar capacidades** al agente con un solo comando.
3. En TI, el mayor retorno está en tareas repetitivas de criterio: auditorías, revisiones y documentación.

> Los gráficos se exportan como `lenguajes.png` y `riesgos.png` en la carpeta del cuaderno.